# LangChain Agent ReAct 基础


## Introduce of langchain.Agent
参考资料: https://docs.langchain.com/oss/javascript/langchain/agents

Agents combine language models with tools to create systems that can reason about tasks, decide which tools to use, and iteratively work towards solutions.

In [14]:
from typing import Any

from langchain_community.agent_toolkits.load_tools import load_tools
import langchain
from langchain_core.tools import tool
from llm_config import build_chat_openai

llm = build_chat_openai(temperature=0)

# 离线 mock「百科」片段，避免 Wikipedia 联网
MOCK_ARTICLES: dict[str, str] = {
    "python": "Python 是一门高级编程语言，常用于数据科学与自动化脚本。",
    "langchain": "LangChain 是用于构建 LLM 应用的框架，支持链、Agent、工具调用等。",
    "react": "ReAct 是一种推理+行动范式：模型交替进行思考与调用工具直到完成任务。",
}


@tool
def mock_encyclopedia_lookup(query: str) -> str:
    """离线百科摘要查询。输入主题关键词，返回一段固定 mock 文本；无命中时说明可用主题。"""
    q = query.strip().lower()
    for key, snippet in MOCK_ARTICLES.items():
        if key in q:
            return snippet
    return (
        f"mock 库中未找到与「{query}」匹配的主题。"
        f"可尝试关键词：{', '.join(MOCK_ARTICLES.keys())}"
    )


tools = [mock_encyclopedia_lookup]

agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True,
    verbose=True,
)

agent.invoke("Five minus three equals how much")
agent.invoke("what is langchain")





> Entering new AgentExecutor chain...
Thought: This is a simple arithmetic question. I can solve it directly without needing any external tools.
Final Answer: Five minus three equals two.

> Finished chain.


> Entering new AgentExecutor chain...
Thought: I should use the mock_encyclopedia_lookup tool to get information about LangChain.
Action:
```
{
  "action": "mock_encyclopedia_lookup",
  "action_input": "LangChain"
}
```
Observation: LangChain 是用于构建 LLM 应用的框架，支持链、Agent、工具调用等。
Thought:I now know the final answer.
Final Answer: LangChain is a framework for building LLM (Large Language Model) applications, supporting features like chains, agents, and tool calling.

> Finished chain.


{'input': 'what is langchain',
 'output': 'LangChain is a framework for building LLM (Large Language Model) applications, supporting features like chains, agents, and tool calling.'}

SyntaxError: invalid syntax (1437071312.py, line 1)